# 01 Macro model votes

Each endpoint is fetched once. These stored readings describe model evidence, not investment advice.

Each panel refreshes on a schedule; each panel shows its source date. Free access grants no redistribution rights. **Do not republish the data you fetch.**

In [ ]:
import json
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'notebooks' / '_shared.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'notebooks'))
sys.path.insert(0, str(ROOT / 'python' / 'src'))
from _shared import fetch_panel, get_api_key, make_client, nested, selected
client, catalogue = make_client()
api_key = get_api_key()

In [ ]:
model_columns = {
    'as_of_date': 'The observation date stored with the diagnostics.',
    'score': 'The stored composite model score.',
    'state': 'The stored interpretation of the model readings.',
    'regime_confirmation': 'The stored regime-confirmation state.',
    'votes_json': 'The stored vote object, encoded as JSON without changing its values.',
}
def model_rows(payload):
    data = payload.get('data') or {}
    return [{'as_of_date': nested(data, 'source', 'as_of_date'), 'score': data.get('score'), 'state': data.get('state'), 'regime_confirmation': data.get('regime_confirmation'), 'votes_json': json.dumps(data.get('votes') or {}, sort_keys=True)}]
model_frame = fetch_panel('Stored macro model votes', lambda: client.member_macro_model_votes(api_key), model_rows, model_columns, source_date=lambda p: nested(p, 'data', 'source', 'as_of_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'message'))

In [ ]:
index_columns = {
    'index_id': 'The index associated with the voter readings.',
    'voter': 'The statistical method that supplied the reading.',
    'vote': 'The stored numeric vote, with null for an unavailable reading.',
    'fitted_value': 'The stored fitted value from the voter.',
    'probability': 'The stored probability when the method provides one.',
    'oos_hit_rate': 'The producer summary for prior out-of-sample directional matches.',
    'oos_se': 'The standard error attached to that producer summary.',
    'oos_n': 'The number of observations behind that producer summary.',
    'brier': 'The stored probability-scoring measure when supplied.',
    'majority': 'The equal-weight majority across the six voters.',
    'as_of_date': 'The shared observation date for the voter snapshot.',
}
def index_rows(payload):
    data = payload.get('data') or {}
    rows = []
    for index in data.get('indexes') or []:
        for voter in index.get('voters') or []:
            rows.append({**selected(voter, ('voter', 'vote', 'fitted_value', 'probability', 'oos_hit_rate', 'oos_se', 'oos_n', 'brier')), 'index_id': index.get('index_id'), 'majority': (index.get('tally') or {}).get('majority'), 'as_of_date': data.get('as_of_date')})
    return rows
index_frame = fetch_panel('Stored index voter readings', lambda: client.member_macro_index_votes(api_key), index_rows, index_columns, source_date=lambda p: nested(p, 'data', 'as_of_date'), status=lambda p: nested(p, 'data', 'status'), message=lambda p: nested(p, 'data', 'message'))

In [ ]:
cycle_names = ('market', 'date', 'leading_index', 'coincident_index', 'lagging_index', 'published_score', 'pc1_score', 'eqw_score', 'n_used', 'n_max', 'method', 'methodology_version')
cycle_columns = {
    'market': 'The market associated with the cycle reading.',
    'date': 'The observation date for the cycle reading.',
    'leading_index': 'The stored composite for indicators that tend to move earlier.',
    'coincident_index': 'The stored composite for indicators of current conditions.',
    'lagging_index': 'The stored composite for indicators that tend to move later.',
    'published_score': 'The cycle score selected for publication.',
    'pc1_score': 'The score from the first principal component when supplied.',
    'eqw_score': 'The equal-weight component score when supplied.',
    'n_used': 'The number of inputs used in the reading.',
    'n_max': 'The maximum number of inputs in the defined panel.',
    'method': 'The method selected for the published reading.',
    'methodology_version': 'The methodology identifier stored with the row.',
}
cycle_frame = fetch_panel('Economic cycle composites', lambda: client.member_macro_cycle_composites(api_key), lambda p: [selected(row, cycle_names) for row in p.get('rows') or []], cycle_columns, source_date=lambda p: max((row.get('date') for row in p.get('rows') or [] if row.get('date')), default=None), status=lambda p: 'not supplied; rows returned' if p.get('rows') else 'not supplied; empty', message=lambda p: None if p.get('rows') else 'No cycle rows returned.')

In [ ]:
recession_names = ('market', 'date', 'recession_prob_1m', 'recession_prob_3m', 'recession_prob_12m', 'target_source', 'methodology_version')
recession_columns = {
    'market': 'The market associated with the probability row.',
    'date': 'The observation date for the probability row.',
    'recession_prob_1m': 'The stored probability for the one-month horizon.',
    'recession_prob_3m': 'The stored probability for the three-month horizon.',
    'recession_prob_12m': 'The stored probability for the 12-month horizon.',
    'target_source': 'The target series or proxy used by the model.',
    'methodology_version': 'The methodology identifier stored with the row.',
}
recession_frame = fetch_panel('Recession probabilities', lambda: client.member_macro_recession_probability(api_key), lambda p: [selected(row, recession_names) for row in p.get('rows') or []], recession_columns, source_date=lambda p: max((row.get('date') for row in p.get('rows') or [] if row.get('date')), default=None), status=lambda p: 'not supplied; rows returned' if p.get('rows') else 'not supplied; empty', message=lambda p: None if p.get('rows') else 'No recession-probability rows returned.')

In [ ]:
hmm_names = ('market', 'date', 'regime_label', 'prob_expansion', 'prob_late_cycle', 'prob_contraction', 'methodology_version')
hmm_columns = {
    'market': 'The market associated with the regime row.',
    'date': 'The observation date for the regime row.',
    'regime_label': 'The regime label assigned by the stored classifier.',
    'prob_expansion': 'The stored probability assigned to expansion.',
    'prob_late_cycle': 'The stored probability assigned to late cycle.',
    'prob_contraction': 'The stored probability assigned to contraction.',
    'methodology_version': 'The methodology identifier stored with the row.',
}
hmm_frame = fetch_panel('Stored macro regime probabilities', lambda: client.member_fund_hmm_regime(api_key), lambda p: [selected(row, hmm_names) for row in p.get('rows') or []], hmm_columns, source_date=lambda p: max((row.get('date') for row in p.get('rows') or [] if row.get('date')), default=None), status=lambda p: p.get('source_mode') or 'not supplied', message=lambda p: p.get('message') or (None if p.get('rows') else 'No regime rows returned.'))